# Univariate Gaussian Score

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

PDF, log PDF tangents, and score s(x) for a 1D Gaussian.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/gaussian-score-1d/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch
from torch.distributions import Normal

# The demo's defaults: p = N(mu, sigma^2)
mu, sigma = 2.0, 1.5
p = Normal(torch.tensor(mu, dtype=torch.float64), torch.tensor(sigma, dtype=torch.float64))

# ── The score is the slope of log p: s(x) = d/dx log p(x) = -(x - mu) / sigma^2
x = torch.tensor([-2.0, 0.0, 1.0, 3.0, 4.0, 6.0], dtype=torch.float64, requires_grad=True)
(score_autograd,) = torch.autograd.grad(p.log_prob(x).sum(), x)
score_formula = -(x.detach() - mu) / sigma ** 2

print("   x     score (autograd)   -(x - mu)/sigma^2")
for xi, a, f in zip(x.tolist(), score_autograd.tolist(), score_formula.tolist()):
    print(f"{xi:5.1f}     {a:+.4f}             {f:+.4f}")
# Positive left of the mean, negative right of it: the score points uphill, toward mu
print("score at the mean:", -(mu - mu) / sigma ** 2)

# ── Two facts used later: E_p[s(X)] = 0 and E_p[s(X)^2] = 1/sigma^2 (Fisher information)
torch.manual_seed(0)
xs = p.sample((500000,))
s = -(xs - mu) / sigma ** 2
print(f"\nE[s(X)] = {s.mean().item():+.5f} (0),  E[s(X)^2] = {(s ** 2).mean().item():.5f} (1/sigma^2 = {1 / sigma ** 2:.5f})")

# The score does not need the normalizing constant: log(c p) has the same slope
(scaled,) = torch.autograd.grad((p.log_prob(x) + 7.0).sum(), x)
print("score of 'p times a constant' is unchanged:", torch.allclose(scaled, score_autograd))

   x     score (autograd)   -(x - mu)/sigma^2
 -2.0     +1.7778             +1.7778
  0.0     +0.8889             +0.8889
  1.0     +0.4444             +0.4444
  3.0     -0.4444             -0.4444
  4.0     -0.8889             -0.8889
  6.0     -1.7778             -1.7778
score at the mean: -0.0

E[s(X)] = +0.00011 (0),  E[s(X)^2] = 0.44345 (1/sigma^2 = 0.44444)
score of 'p times a constant' is unchanged: True
